In [358]:
import pandas as pd
import numpy as np
import warnings
import re

In [362]:
cell_table.fov

0          Leap001_10
1          Leap001_10
2          Leap001_10
3          Leap001_10
4          Leap001_10
              ...    
3964191     Leap096_9
3964192     Leap096_9
3964193     Leap096_9
3964194     Leap096_9
3964195     Leap096_9
Name: fov, Length: 3964196, dtype: object

,cell_size,Alpha-SMA,B7-H4,Beta-Catenin,CD107a,CD11b,CD14,CD16,CD163,CD20,...,major_minor_axis_ratio_nuclear,perim_square_over_area_nuclear,major_axis_equiv_diam_ratio_nuclear,convex_hull_resid_nuclear,centroid_dif_nuclear,num_concavities_nuclear,nc_ratio_nuclear,fov,cell_meta_cluster,qc_pass
0,20.0,9.719218,2.562145,0.841014,2.111613,6.085201,4.050718,1.285438,1.266174,0.078957,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.0,0.000000,Leap001_10,Immune cell / Fibroblast,False
1,23.0,2.024557,3.809946,0.659182,7.519738,5.597602,7.389095,2.442724,4.658532,0.088697,...,1.732051,5.422792,1.227992,0.00000,0.000000,0.0,0.217391,Leap001_10,Immune cell / Fibroblast,False
2,28.0,0.522853,1.674286,0.237839,0.242442,1.747657,1.450553,0.269625,0.084811,0.047978,...,2.765529,13.184993,1.640943,0.00000,0.000000,0.0,0.821429,Leap001_10,Neutrophil / B7-H4,True
3,17.0,1.718866,4.476247,0.979894,9.853949,7.179607,11.453550,2.812294,2.210392,0.035725,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.0,0.000000,Leap001_10,Immune cell / Fibroblast,False
4,22.0,0.896025,1.593269,0.428365,0.260594,2.662433,1.387356,0.056901,0.078901,0.034602,...,2.366055,11.572750,1.515735,0.00000,0.000000,0.0,0.863636,Leap001_10,Immune cell / Fibroblast,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3964191,67.0,3.106543,2.539710,4.099438,0.621717,1.340995,1.641260,0.373170,0.537264,0.068023,...,2.801223,15.035385,1.671491,0.02381,0.016370,0.0,0.611940,Leap096_9,Activated T-cell,True
3964192,25.0,1.757703,1.206490,2.078473,1.073784,0.748328,1.029817,0.012845,0.047101,0.036252,...,1.369306,8.333333,1.144114,0.00000,0.000000,0.0,0.480000,Leap096_9,Immune cell / Fibroblast,True
3964193,27.0,1.821112,2.317336,4.140246,0.554483,4.589223,1.319677,0.188056,0.110803,3.941155,...,2.015125,10.971236,1.414819,0.00000,0.000000,0.0,0.555556,Leap096_9,Mem B-cell,True
3964194,17.0,27.706037,3.473785,7.752644,0.923455,1.022657,1.662537,0.334678,0.452365,0.053718,...,2.132889,9.003672,1.451259,0.06250,0.032275,0.0,0.882353,Leap096_9,Activated T-cell,True


In [ ]:

def split_text(text):
    '''
    Text is match at the beginning and the string is shortened repeatedely
    '''
    regex = r'^\d{1,3}=core|resection'
    text = text.replace(' ','')#remove any space in the string
    data = {}
    while True:
        a = re.search(regex,text,flags=re.IGNORECASE)
        if a is None:
            break
        temp =  text[:a.end()]
        key,value = temp.split('=')
        data[int(key)] = value
        text= text[a.end():]
    return data
def unravel_response_df(response):     
    '''Transform the response Excel file to machine readable'''      
    def process_row(row):
        def check_keywords(input_text):
            keywords = ["bilateral", "same patient"]
            for keyword in keywords:
                if keyword.upper() in input_text.upper():
                    return True
            return False

        leap_id = row.LEAP_ID
        prefix = 'LEAP'
        num_part = leap_id.replace(prefix,'')#remove LEAP
        parts = num_part.split('/')
        if len(parts) > 1:
            #if comments distinguis between core/section
            if ('core'.upper() in row.COMMENTS.upper()) or ('resection'.upper() in row.COMMENTS.upper()):  
                if len(parts) == row.COMMENTS.count('='):
                    #the number of = characters  matches the number of split in the LEAP_ID column, that is the normal cases
                    #key is leap number, value is the corresponding comment (core or resection)
                    mapper = split_text(row.COMMENTS)#transform mapper to dict and make the key an integer to help indexing of the dictionary below
                elif (('core'.upper() in row.COMMENTS.upper()) ^('resection'.upper() in row.COMMENTS.upper())):
                    # we are in the case where we have more Leaps that information about CORE/RESECTION.
                    #If Comments column contains either CORE or Resection, but not both, this will be assigned to all leaps

                    if 'core' in row.COMMENTS:
                        name = 'core'
                    else:
                        name = 'resection'
                    mapper = {}
                    for part in parts:
                        ind = int(part)
                        mapper[ind] = name.upper()
                else:
                    raise ValueError('Multiple Leap ID do not correspond but not the same number of info in COMMENTS column.\n Failed to interpret the comment: ', row.COMMENTS)
                
            else:
                raise ValueError('Failed to interpret the comment: ', row.COMMENTS)
            try:

                new_rows= []
                for part in parts:
                    #split the row into two
                    new_row = row.copy()
                    new_row.LEAP_ID = prefix+"%03d" %int(part)
                    ind = int(part)
                    new_row.COMMENTS = mapper[ind].upper()
                    new_rows +=[new_row]
            except KeyError:
                raise KeyError('Failed to interpret the comment: ', row.COMMENTS, 'Cannot find key',ind)
            return new_rows
        else:
            if 'CORE' in row.COMMENTS.upper():
                name = 'core'
            elif 'RESECTION' in row.COMMENTS.upper():
                name = 'resection'
            else:
                raise ValueError('Failed to interpret the comment: ', row.COMMENTS)
            row.LEAP_ID = prefix+"%03d" %int(num_part)
            row.COMMENTS = name.upper()
            return [row]

    transformed = []
    for _,el in response.iterrows():

        transformed+=[process_row(el)]
    return pd.DataFrame([b for el in transformed for b in el])


In [357]:
path0 = '../'
response = pd.read_excel(path0+"IMC_data//ExtraDocs/LEAP code response data 05122023.xlsx")
response.columns = response.columns.str.replace(' ','_')#use underscore rather than space
response.COMMENTS = response.COMMENTS.str.replace(',',' ')#the file mix space and , as seprators, using only whitespace for simplicity
response = unravel_response_df(response[['LEAP_ID', 'Response', 'COMMENTS']])

biosamples =pd.read_excel(path0+'IMC_data/ExtraDocs/LEAP biobank samples.xlsx')
biosamples.columns = biosamples.columns.str.replace(' ','_')#use underscore rather than space
biosamples.drop('OUTCOME____(RESPONDER,_NON-RESPONDER)',inplace = True,axis = 1)# response status is in response
metadata = biosamples.merge(response,on='LEAP_ID')
#check that the core/resection of the 2 dataframes is consistent whenever data are provided
cond1 = ~metadata['SAMPLE_TYPE_(CORE/RESECTION)'].isna()
cond2 = ~metadata['COMMENTS'].isna()

if not (metadata['SAMPLE_TYPE_(CORE/RESECTION)']==metadata['COMMENTS'].str.upper())[cond1*cond2].all():
    warnings.warn('The response data and biobank sample dataframes have different core/resection annotation')
metadata.to_csv(path0+'IMC_data//ExtraDocs/processed_response.csv',index = False)


In [355]:
cond = metadata['SAMPLE_TYPE_(CORE/RESECTION)'].isna()
metadata.loc[cond,'SAMPLE_TYPE_(CORE/RESECTION)'] = metadata.COMMENTS[cond]